# Setup (10 minutes)

Spend the first 10 minutes for the setup to:

1. Replace CID in the file name with your CID, e.g. 123456_Coursework2_Part2.ipynb
2. Read and execute the following sections:
- **[S1](#s1). Package Imports** <a name="index-s1"></a>
- **[S2](#s2). Queries on Documentation** <a name="index-s2"></a>
- **[S3](#s3). Dataset Loading** <a name="index-s3"></a>
3. Start to search and open lecture notes and notebooks that you will need to complete the tasks.

<a name="s1"></a>

## S1. Package Imports [(index)](#index-s1)

In [4]:
import numpy as np
import pandas as pd
from numpy import linalg
import matplotlib.pyplot as plt
from heapq import heappush, heappop, heapify
#from tqdm import tqdm

# we define a mix of fontsizes, for different parts of a plot
SMALL_SIZE = 12
MEDIUM_SIZE = 16
BIGGER_SIZE = 20

# example of how you can use these fontsizes to set a global configuration for matplotlib;
# you should assign them based on the specific appearance of the figures you are producing
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)    # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=MEDIUM_SIZE)   # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

<a name="s2"></a>

## S2. Queries on Documentation [(index)](#index-s2)

Documentation of common Python libraries (*e.g.*, np.linalg.eig) can be accessed locally by executing the following expression in an arbitrary cell.

In [5]:
# Example: to get the documentation of `np.linalg.eig` function for Eigen decomposition.
# Press the [x] button to close the documentation dialogue.

?np.linalg.eig

Signature:       np.linalg.eig(a)
Call signature:  np.linalg.eig(*args, **kwargs)
Type:            _ArrayFunctionDispatcher
String form:     <function eig at 0x10ad3c4a0>
File:            ~/Documents/Y3_Imperial/data science/venv/lib/python3.11/site-packages/numpy/linalg/_linalg.py
Docstring:      
Compute the eigenvalues and right eigenvectors of a square array.

Parameters
----------
a : (..., M, M) array
    Matrices for which the eigenvalues and right eigenvectors will
    be computed

Returns
-------
A namedtuple with the following attributes:

eigenvalues : (..., M) array
    The eigenvalues, each repeated according to its multiplicity.
    The eigenvalues are not necessarily ordered. The resulting
    array will be of complex type, unless the imaginary part is
    zero in which case it will be cast to a real type. When `a`
    is real the resulting eigenvalues will be real (0 imaginary
    part) or occur in conjugate pairs

eigenvectors : (..., M, M) array
    The normalized (un

<a name="s3"></a>

## S3. Dataset Loading [(index)](#index-s3)

#### Dataset description (just for reference, it's a summary from Part 1)
In Coursework 2, you are working with a dataset on **gene expression**, contained in the file `gene_expression_transcriptomic_data.csv`.

The dataset is a single-cell transcriptomic dataset, which measures the level of gene expression for different genes in terms of the abundance of RNA that has been transcribed from the corresponding gene inside single cells (our samples). Gene expression levels are reported in terms of RNA molecular concentration in $\log_2$ scale. The column `Treatment` contains binary classification labels ($0,1$): $0$ indicates that the cell was not subject to a drug treatment, $1$ indicates that it was.  

Here in In Part 2, like you did in Task 2 of Part 1, you will use **only the first 400 samples** and **only the subset of 10 genes** known to be most correlated to MAP7, whose expression levels are reported **in the first 10 columns** of `gene_expression_transcriptomic_data.csv`. Hence, the dataset to use here is a matrix $\mathbf{X}_{(N \times p)}$ with $N=400$ and $p=10$. You will need also the corresponding list of 400 binary labels under the column `Treatment`.

You need to standardise the dataset matrix $\mathbf{X}$.

These steps of data uploading, data extraction, and standardisation are already performed for you in the next cell.

In [6]:
PATH=''

# Dataset upload #
dataset = pd.read_csv(PATH + "gene_expression_transcriptomic_data.csv")

X = dataset.iloc[:400, :10].values # extract the first 400 rows and 10 columns
treatment = dataset.iloc[:, 101].values[:400, ] # extract the corresponding 400 treatment labels

# check their shapes
print(X.shape)
print(treatment.shape)

(400, 10)
(400,)


In [7]:
def standardise(X):
    """
    Standardise features.

    Parameters:
        X (np.array): Feature matrix.

    Returns:
        X_std (np.array): Standardised feature matrix
    """

    mu = np.mean(X, axis=0, keepdims=True)
    sigma = np.std(X, axis=0, keepdims=True)
    X_std = (X - mu) / sigma

    return X_std


## Data standardisation ##
X_std = standardise(X)

**Note:** work only with `X_std` from now on.

## Coursework 2 Part 2 (120 minutes)

###  Tasks Outline

- [Task 3 (25 marks)](#task-3): Kernel Principal Component Analysis
- [Task 4 (25 marks)](#task-4): Logistic Regression
- [Task 5 (20 marks)](#task-5): Hierarchical Clustering

<a name="task-3"></a>
# Task 3 (25 marks): Kernel Principal Component Analysis ([index](#outline))


In this task you will perform dimensionality reduction on the (standardised) dataset `X_std` via kernel Principal Component Analysis (kernel PCA).

**3.1 - Kernel Matrix (12 marks).** As first step, you need to compute the Gram matrix of kernels to use in Kernel PCA. Complete the two following skeleton codes to compute the Gram matrix with, respectively:
- a polynomial kernel of degree up to $n$ that, for two vectors $\mathbf{x}$ and $\mathbf{y}$, is defined as:
$$
k(\mathbf{x},\mathbf{y}) = (\mathbf{x}\cdot\mathbf{y} + 2.5)^n
$$
- a radial kernel with parameter $c$ that, for two vectors $\mathbf{x}$ and $\mathbf{y}$, defined as:
$$
k(\mathbf{x},\mathbf{y}) = e^{-\frac{||\mathbf{x} - \mathbf{y}||^2}{c}}
$$

In [8]:
def polynomial_kernel(X1, X2, n):
    """
    Compute the Gram matrix with a polynomial kernel between two input arrays.

    Parameters:
      X1 (np.ndarray): first input array, shape (N, p).
      X2 (np.ndarray): second input array, shape (M, p).
      n (float): the degree of the polynomial.

    Returns:
      final_kernel (np.ndarray): centred Gram matrix, shape (N, M).
    """

    assert X1.shape[1] == X2.shape[1] # expect X1 and X2 to have same number of columns

    N, p = X1.shape
    M, p = X2.shape
    kernel = np.zeros((N, M))

    # Compute the Gram kernel matrix using the polynomial kernel defined above
    for i in range(N):

        inner_product = np.ones(M) # <-- EDIT THIS LINE

        kernel[i, :] = np.ones(M) # <-- EDIT THIS LINE

    # Centre the Gram kernel matrix
    iden = 1 / N * np.ones((N, N))

    final_kernel = np.ones((N, N)) # <-- EDIT THIS LINE

    return final_kernel

In [9]:
def radial_kernel(X1, X2, c):
    """
    Compute the Gram matrix with a radial kernel between two input arrays.

    Parameters:
      X1 (np.ndarray): first input array, shape (N, p).
      X2 (np.ndarray): second input array, shape (M, p).
      c (float): scaling coefficient at the exponent of the radial kernel.

    Returns:
      final_kernel (np.ndarray): centred Gram matrix, shape (N, M).
    """

    assert X1.shape[1] == X2.shape[1] # expect X1 and X2 to have same number of columns

    N, p = X1.shape
    M, p = X2.shape
    kernel = np.zeros((N, M))

    # Compute the Gram kernel matrix using the radial kernel defined above
    for i in range(N):

        inner_product = np.ones(M) # <-- EDIT THIS LINE

        kernel[i, :] = np.ones(M) # <-- EDIT THIS LINE

    # Centre the Gram kernel matrix
    iden = 1 / N * np.ones((N, N))

    final_kernel = np.ones((N, N)) # <-- EDIT THIS LINE

    return final_kernel

**(Continue 3.1)**

Starting from the dataset `X_std`, compute the polynomial kernel Gram matrix with $n=4$ and the radial kernel Gram matrix with $c=10$. For each matrix, produce a plot of its eigenvalue spectrum. Explain why this plot allows you to verify that you computed valid kernels.


**3.2 - Kernel PCA projections (13 marks).** Write a function `pca_function_kernel` to compute the data projection onto the first $m$ non-linear components of Kernel PCA by completing the following skeleton code. The function should be able to work for any Gram kernel matrix specified via the `kernel` argument. [Hint: some lines can be the same as from the course's notebooks!]

In [10]:
def pca_function_kernel(X, m, kernel, alpha=1):
    """
    Compute the projection of input data using Kernel PCA.

    Parameters:
      X (np.ndarray): input array, shape (N, p).
      m (int): number of non-linear principal components.
      kernel (callable): function to be used as Gram kernel matrix.
      alpha (float): parameter of kernel function (e.g. n for polynomial, c for radial).
    Returns:
      projection (np.ndarray): data projection matrix, shape (N, m).
    """

    # matrix to diagonalise
    C = np.ones((X.shape[0], X.shape[0])) # <-- EDIT THIS LINE

    # Computing the eigenvalues and eigenvectors
    eigen = np.ones((X.shape[0], X.shape[0])) # <-- EDIT THIS LINE

    # extract the m top-ranking eigenvalues
    eigenvalues = np.ones(m) # <-- EDIT THIS LINE

    # avoid too small numerical values
    threshold = 1e-10
    eigenvalues[np.abs(eigenvalues) < threshold] = 1e-7

    # and the corresponding eigenvectors
    eigenvectors = np.ones((X.shape[0], m)) # <-- EDIT THIS LINE

    # obtain the data projection
    projection = X[:, 0:m] # <-- EDIT THIS LINE

    return projection

**(Continue 3.2)**

Compute the projection of the dataset `X_std` onto the first $m=2$ non-linear components using the polynomial kernel function with $n=3, 4$ and the radial kernel function with $c=5,10$ that you defined in **3.1**. Produce a scatter plot of the projection onto the first component against the projection onto the second, for each kernel type and each value of $n$ and $c$ (i.e., 4 plots), colouring each data point according to the `Treatment` label.

<a name="task-4"></a>
# Task 4 (25 marks): Logistic Regression ([index](#outline))

In this task you will assess the extent to which the lower-dimensional PCA projection reflects the partition into the two `Treatment` classes, comparing kernel PCA to standard linear PCA.

**4.1 - Linear PCA (5 marks).** Using code from the Week 8 notebook, perform linear PCA on the dataset `X_std` to obtain the data projections onto the first $m=2$ principal components. Produce a scatter plot of these projections, like in **3.2**, colouring each data point according to the `Treatment` label.


**4.2 - Logistic Regression on linear PCA (15 marks).**

**4.2.1** Taking code from the logistic regression notebook (Week 2), train a logistic regression classifier using as training set the projections of the *entire* dataset `X_std` on the first $m=2$ principal components of linear PCA from **4.1**. [Hint: the training set has then size $400\times2$]. Report the accuracy on the training set (as is computed inside the function `model` from the logistic regression notebook), commenting on your result.

**Note:** If you see a `RuntimeWarning` message during training, fix it using one of the strategies we have seen during the course to avoid computing numerically the logarithm of zero.


**4.2.2** Discretise the predictions of the classifier from **4.2.1**, using 3 different classification thresholds: $\tau=0.5$, $\tau=0.6$, $\tau=0.7$. For each value of $\tau$: compute and report the precision on the training set; draw the classifier's decision boundary on the plot that you produced in **4.1** of the PCA data projections. Discuss your results.


**4.3 - Logistic Regression on kernel PCA (5 marks).** Repeat subtask **4.2.1** using as training set the projections of the *entire* dataset `X_std` on the first $m=2$ principal components of kernel PCA from **3.2**, both for the polynomial kernel with $n=3$ and the radial kernel with $c=5$. Discuss your results.

[Note: If you did not obtain a usable output from **3.1** and **3.2**, use the default output of our skeleton functions].


<a name="task-5"></a>
# Task 5 (20 marks): Hierarchical clustering ([index](#outline))

In this task you will perform hard clustering of the samples in `X_std` via Hierarchical Clustering (HC), comparing two linkage choices.

**5.1 - Hard clustering with HC (14 marks).** Using code from Week 7 notebooks, apply HC to cluster the samples in `X_std` using: the Euclidean distance as your pairwise distance metric $D(\mathbf{x}^{(i)}, \mathbf{x}^{(j)})$ between samples $\mathbf{x}^{(i)}$ and $\mathbf{x}^{(j)}$; *complete* linkage.

Let's recall that, in clustering, the between-cluster distance $B(C)$ is defined as:
$$
B(C) = \frac{1}{2} \sum_{k=1}^K \sum_{i \in  c_k}\sum_{j  \notin c_k} D(\mathbf{x}^{(i)}, \mathbf{x}^{(j)})
$$
where $D(\mathbf{x}^{(i)}, \mathbf{x}^{(j)})$, as above, is the pairwise distance between samples used, $K$ is the number of clusters $c_k$ at a certain tree level.

Plot $B(C)$ as a function of the tree levels and use the shape of this plot to determine an optimal number of clusters, justifying your answer.

In [12]:
def pairwise_distances(X):
   '''
   Parameters:
       X (np.ndarray): Samples matrix, shape (N, p).
   Returns:
       distance (np.ndarray): Distance matrix, shape (N, N).
   '''
   N, D = X.shape

  # Apply zero padding with np.zeros and np.concatenate to prevent computing the same values twice.
   distance = np.array([np.concatenate([np.zeros(i + 1), np.sqrt(np.sum((X[i, :] - X[range(i + 1, N), :])**2, axis=1))]) for i in range(N)])

   return distance + distance.T

distance = pairwise_distances(X)

In [18]:
# EDIT THIS FUNCTION

def linkage(distances, labels, i, j):
    """
    This function computes a matrix of distances between the samples of two clusters

    Parameters:
      distances (np.ndarray): Distance matrix, shape (N, N).
      labels (np.ndarray): Cluster index of each sample, shape (N,).
      i (int): The index of the first cluster.
      j (int): The index of the second cluster.
      linkage_type (string): The linkage type to be used,
                             either 'single' or 'average'.
    Returns:
      pairs_distance (np.ndarray): Distance matrix, shape (|c_i|, |c_j|).

    """
    # Select the indices of the first cluster.
    points_i = np.argwhere(labels == i) ## <-- EDIT HERE
    # Select the point indices of the second cluster.
    points_j = np.argwhere(labels == j) ## <-- EDIT HERE
    # Form a cartesian product between the indices in i and indices in j.
    pairs = np.array([[element_i.item(), element_j.item()]  for element_i in points_i for element_j in points_j])
    # Select the pair distances between the points in the two clusters from the distances matrix.
    pairs_distance = distances[pairs[:, 0], pairs[:, 1]]

    return pairs_distance.max()

In [19]:
class Pair:
    """
    A pair of clusters.
    """
    def __init__(self, i: int, j: int, distance: float):
        """
        Args:
            i (int): The index of the first cluster.
            j (int): The index of the second cluster.
            distance (float): The distance between the two clusters.
        """
        self.i = i
        self.j = j
        self.distance = distance

    def __lt__(self, other):
        """
        Compare two pairs of clusters based on their distance.
        less-than definition.
        Args:
            other (Pair): The other pair of clusters.
        Returns:
            bool: True if this pair is less than the other pair.
        """
        return self.distance < other.distance

    def __repr__(self):
        """string representation"""
        return f'({self.i}, {self.j}, {self.distance})'

In [20]:
# EDIT THIS FUNCTION

def hierarchical_clustering(X, distances):
    """
    The agglomerative hierarchical clustering algorithm start with every point as a single
    cluster and each iteration merges two clusters into one. We store all the
    intermediate clustering results with respect to the number of clusters left.

    Parameters:
      X (np.ndarray): Samples matrix, shape (N, p).
      distances (np.ndarray): Distance matrix, shape (N, N).
      linkage_type (string): The linkage type to be used, either 'single', or 'average'.

    Returns:
     labels (np.ndarray):  Cluster labels at each level, shape (N, N).
          Element (i, j) denotes the cluster label of sample i at level j.
    """

    N, D = X.shape
    labels = np.zeros((N, N))

    # Begin with every point in its own cluster
    current_labels = np.arange(N) # <-- EDIT HERE

    # The id to be assigned for the next merged cluster
    next_cluster_id = N

    priority_queue = []
    # Initialise Priority Queue
    for i in range(N):
        for j in range(i + 1, N):
            priority_queue.append(Pair(i, j, distances[i, j]))
    heapify(priority_queue)

    labels[N - 1] = current_labels # the leaves
    # Begin from level (N - 1) to level 1
    for level in range(N - 2, 0, -1):
        # pop and return the smallest item in priority queue
        # i.e. the pair (i, j) of clusters which are closest together
        next_pair = heappop(priority_queue)

        # Remove pair of clusters in the queue which have i and j in them i.e. (i, j) or (j, i)
        priority_queue = [p for p in priority_queue if next_pair.i not in (p.i, p.j) and next_pair.j not in (p.i, p.j)]

        # re sort the queue
        heapify(priority_queue)

        # Merge all samples which currently belong to cluster i or j
        current_labels[(current_labels == next_pair.i) | (current_labels == next_pair.j)] = next_cluster_id

        # Compute the merging cost of the new cluster with all existing clusters to the queue
        for i in set(p.i for p in priority_queue) | set(p.j for p in priority_queue):
            d = linkage(distances, current_labels, next_cluster_id, i) ## <- EDIT HERE
            heappush(priority_queue, Pair(i, next_cluster_id, d))

        next_cluster_id += 1
        # Store the current cluster assignment into the assignments array.
        labels[level, :] = current_labels ## <- EDIT HERE

        # Print at what level the iteration is at.
        if level % 50 == 0:
          print('Merging at level: ', level)

    # add the root labels
    labels[0, :] = next_cluster_id * np.ones(N)
    return labels

h_clustering = hierarchical_clustering(X, distance)

Merging at level:  350
Merging at level:  300
Merging at level:  250
Merging at level:  200
Merging at level:  150
Merging at level:  100
Merging at level:  50


In [21]:
print(h_clustering)

[[798. 798. 798. ... 798. 798. 798.]
 [794. 797. 797. ... 794. 797. 797.]
 [794. 796. 796. ... 794. 796. 795.]
 ...
 [  0.   1.   2. ... 397. 398. 399.]
 [  0.   1.   2. ... 397. 398. 399.]
 [  0.   1.   2. ... 397. 398. 399.]]


In [25]:
def between_cluster_distance(distances, h_clustering):
    dists = []
    K = h_clustering.shape[0]
    for k in range(K-1, 1, -1):
            labels = h_clustering[k]
            B = 0
            for i in range(K):
                  for j in range(i+1, K):
                    B += linkage(distances, labels, i, j)
            dists.append(B)
    return dists

In [26]:
dists = between_cluster_distance(distance, h_clustering)


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

**(Continue 5.1)**

Produce a scatter plots of the first and third data features (i.e., `X_std[:,0]` against `X_std[:,2]`) colouring each data point according to its cluster assignment for the optimal number of clusters that you have just determined.

**5.2 - Linkage type comparison (6 marks).** Repeat task **5.1** with *average* linkage. Using code from the Week 9 notebook, compute and report the Normalised Variation of Information (NVI) between the clustering obtained here (with its optimal number of clusters) and the clustering obtained in **5.1** (with its optimal number of clusters). Discuss the comparison between *complete* and *average* linkage based on the NVI and the scatter plots you have produced.